# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and references are always by their `@id` fields in this notebook.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the croissant dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name} (Version: {metadata.version})\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant metadata to enumerate all record sets defined in the dataset, referencing by `@id` only as required for robust downstream code.

In [ ]:
# List all record sets in the dataset (by their `@id`)
record_set_objs = list(getattr(metadata, 'recordSet', []))
if not record_set_objs:
    print("No record sets found in metadata.")
else:
    print(f"Found {len(record_set_objs)} record set(s):\n")
    for i, rs in enumerate(record_set_objs):
        print(f"{i+1}. @id: {rs['@id']}\n   Name: {rs.get('name', None)}\n   Description: {rs.get('description', None)}\n")
        print("   Fields (by @id):")
        for field in rs.get('field', []):
            # Each field is a dict with keys '@id', 'name', 'description', etc.
            print(f"      - @id: {field['@id']}, name: {field.get('name', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** All references to record sets and fields must be by their `@id`.

In [ ]:
# Build a list of all record set @id's for extraction
record_sets = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]

dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print("  No records found for this record set.")
# Select a representative record set for EDA (use the first by default)
if record_sets:
    example_record_set_id = record_sets[0]
else:
    example_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data, and grouping.

Operations here are **always by field `@id` only**.

In [ ]:
from pandas.api.types import is_numeric_dtype

if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    print(f"Columns in record set ({example_record_set_id}): {df.columns.tolist()}")
    # Find a likely numeric field by heuristics (e.g., look for 'count', 'MW', 'abundance', or numeric dtype columns)
    numeric_candidates = [col for col in df.columns if is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try casting any col to numeric
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notnull().sum() > 0:
                numeric_candidates.append(col)
    print(f"\nLikely numeric fields: {numeric_candidates}")
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Use the first one
        # Convert to numeric if needed
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        print(f"\nSummary statistics for {numeric_field_id}:")
        print(df[numeric_field_id].describe())

        # Filtering: remove nan, and threshold filtering
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records (where {numeric_field_id} > mean): {filtered_df.shape[0]} rows\n")
        display(filtered_df.head(3))

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

        # Grouping: pick a likely group field
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and not is_numeric_dtype(df[col])]
        group_field_id = group_field_candidates[0] if group_field_candidates else None
        if group_field_id:
            print(f"\nGrouping data by field {group_field_id} and aggregating mean for {numeric_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head(3))
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All references are by field `@id` only.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and example_record_set_id in dataframes and 'filtered_df' in locals():
    # Histogram of the selected numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field is present, plot boxplot
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} distribution by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, overview, and perform basic exploratory analysis of a complex scientific dataset structured as a Croissant package, referencing all elements by their `@id` as per best practices.

- Loaded dataset metadata and identified available record sets and key fields.
- Loaded tabular data for each record set and performed typical EDA operations -- filtering, normalization, and grouping -- strictly by `@id`.
- Visualized numeric fields and grouped data when possible.

This approach can be extended for advanced analysis and machine learning, always tracking every field and record set with its Croissant schema `@id` for full traceability.
